# L3 — Pulire i dati (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/L03_pulizia/L03_soluzioni.ipynb)

## Promemoria

- **Shift + Invio** esegue una cella. **File → Salva una copia in Drive** per non perdere il lavoro.
- Per ogni passo trovi 🔎 **cosa cercare**. Gemini (✨) è disponibile, ma prova prima a capire cosa ti serve.
- ⏱️ Bloccato da più di 10 minuti? Chiedi.
- Se fai un pasticcio: **Runtime → Riavvia ed esegui tutto** riparte dal file originale.

## Contesto

Nella L2 hai trovato diversi problemi in `imprese.xlsx`. Oggi li correggi, uno alla volta.

**La regola di oggi:** dopo ogni correzione, **verifica** che abbia fatto quello che volevi. Conta prima, conta dopo.

## 0. Preparazione

Esegui questa cella così com'è.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/imprese.xlsx"
df = pd.read_excel(url)
df.shape

## Parte 1 — Le righe duplicate

### 1. Quanti duplicati?

Quante righe di `df` sono **identiche** a un'altra riga?

🔎 *Da cercare:* `pandas duplicated · pandas sum`

In [ ]:
df.duplicated().sum()

### 2. Toglili

Elimina le righe duplicate e salva il risultato di nuovo in `df`. Poi verifica: quante righe ha ora? Sono quelle che ti aspettavi?

🔎 *Da cercare:* `pandas drop_duplicates`

In [ ]:
df = df.drop_duplicates()
df.shape

## Parte 2 — Il fatturato

### 3. Quanti "n.d."?

Quante righe hanno il testo `"n.d."` nella colonna `fatturato`? Segnati il numero: ti servirà.

🔎 *Da cercare:* `pandas contare valori uguali`

In [ ]:
(df["fatturato"] == "n.d.").sum()

### 4. Da testo a numero

Trasforma `fatturato` in numero e salvalo di nuovo in `df["fatturato"]`.

Un valore come `"1.032,9"` deve diventare `1032.9`. Pensa a cosa devono diventare il **punto** e la **virgola**, e in che **ordine** fare le sostituzioni. I valori `"n.d."` non sono numeri: devono diventare valori mancanti.

🔎 *Da cercare:* `pandas str.replace · pandas to_numeric errors coerce`

In [ ]:
df["fatturato"] = pd.to_numeric(
    df["fatturato"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce",
)
df["fatturato"].head()

### 5. Verifica

Quanti valori mancanti ha ora la colonna `fatturato`? Deve essere **lo stesso numero** del passo 3. Se è più alto, la conversione ha buttato via dei dati: torna al passo 4.

🔎 *Da cercare:* `pandas isna`

In [ ]:
df["fatturato"].isna().sum()

*Risposta:* 50, come i "n.d.". Chi sostituisce prima la virgola con il punto ottiene `"1.032.9"`, che non è un numero: i valori mancanti diventano più di 1.600, **senza nessun errore**.

### 6. Ha senso?

Calcola il fatturato **mediano** e il **massimo**. Poi mostra le 10 righe con il fatturato più alto. Noti qualcosa di strano?

🔎 *Da cercare:* `pandas median · pandas sort_values ascending`

In [ ]:
print(df["fatturato"].median(), df["fatturato"].max())
df.sort_values("fatturato", ascending=False).head(10)

*Risposta:* Il mediano è circa 1.400 (migliaia di euro), il massimo oltre 24 milioni. Cinque valori sono enormi e poi c'è un salto netto. Guardando la stessa impresa negli altri anni (prova!) si vede che il fatturato è circa mille volte più piccolo: quei valori sono stati inseriti in euro invece che in migliaia.

### 7. Correggi gli errori di unità

Scegli una soglia oltre la quale un fatturato è chiaramente un errore, e dividi per 1.000 quei valori. Scrivi nella cella di testo sotto **perché** hai scelto quella soglia. Poi ricontrolla il massimo.

🔎 *Da cercare:* `pandas loc assegnare con condizione`

In [ ]:
errori = df["fatturato"] > 200_000
print(errori.sum())
df.loc[errori, "fatturato"] = df.loc[errori, "fatturato"] / 1000
df["fatturato"].max()

*Risposta:* Soglia 200.000: tra i cinque valori anomali (oltre 300.000) e il più grande valore plausibile (circa 64.000) c'è un salto netto, e ogni soglia in quel buco dà lo stesso risultato.

## Parte 3 — Le regioni

### 8. Quante regioni?

Quante regioni diverse ci sono nella colonna `regione`? Quante dovrebbero essere?

🔎 *Da cercare:* `pandas nunique`

In [ ]:
df["regione"].nunique()

### 9. Un dizionario

Per correggere i nomi delle regioni ti servirà un **dizionario**: una tabellina di coppie *chiave → valore*, scritta tra parentesi graffe. Esegui la cella qui sotto, poi aggiungi la Puglia (capoluogo: Bari) e stampa il capoluogo della Toscana.

🔎 *Da cercare:* ``python dizionario` · `python dizionario aggiungere chiave``

In [ ]:
capoluoghi = {"Lombardia": "Milano", "Lazio": "Roma", "Toscana": "Firenze"}
capoluoghi["Puglia"] = "Bari"
print(capoluoghi["Toscana"])
print(capoluoghi)

*Risposta:* Con `capoluoghi["Puglia"] = "Bari"` si aggiunge una coppia; con `capoluoghi["Toscana"]` si legge il valore di una chiave. Nel passo successivo `replace` usa un dizionario allo stesso modo: *valore sbagliato → valore giusto*.

### 10. Sistemale

Correggi la colonna `regione` in modo che ogni regione sia scritta in un solo modo. Ci sono spazi di troppo, maiuscole sbagliate e una regione scritta in due modi. Alla fine `nunique()` deve dare 20.

🔎 *Da cercare:* `pandas str.strip · pandas replace`

In [ ]:
df["regione"] = df["regione"].str.strip().replace({
    "lombardia": "Lombardia",
    "LOMBARDIA": "Lombardia",
    "Emilia Romagna": "Emilia-Romagna",
})
df["regione"].nunique()

*Risposta:* Attenzione a `str.title()`: sistema le maiuscole ma trasforma `Valle d'Aosta` in `Valle D'Aosta` e lascia `Emilia Romagna` senza trattino. Risultato: 21 regioni. Una correzione esplicita è più sicura.

## Parte 4 — Addetti e valori mancanti

### 11. Addetti impossibili

Quante righe hanno un numero di addetti **negativo**? Mostra quali sono: di che impresa e di che anno?

🔎 *Da cercare:* `pandas filtrare righe condizione`

In [ ]:
df[df["addetti"] < 0][["id_impresa", "anno", "addetti"]]

### 12. Correggere una cella a mano

Hai scritto al collega, che ti risponde: *"IMP0253 nel 2019 aveva 12 addetti: il segno meno è un errore di battitura."*

Correggi **solo quella cella**. Non usare il numero della riga: individua la riga con l'impresa **e** l'anno. Poi verifica che il valore sia 12 e che non ci siano più addetti negativi.

🔎 *Da cercare:* `pandas loc più condizioni &`

In [ ]:
riga = (df["id_impresa"] == "IMP0253") & (df["anno"] == 2019)
print(riga.sum())
df.loc[riga, "addetti"] = 12
print(df.loc[riga, "addetti"])
print((df["addetti"] < 0).sum())

*Risposta:* La condizione doppia si scrive con `&` e ogni pezzo va tra parentesi. Prima di modificare, `riga.sum()` deve dare 1: una sola cella. Perché non `df.loc[1260, "addetti"] = 12`? Oggi funziona, ma se cambi l'ordine o togli righe, la riga 1260 diventa un'altra impresa: la correzione finisce nel posto sbagliato senza nessun errore.

### 13. Quanti valori mancanti?

Conta i valori mancanti di **ogni colonna**, con un solo comando.

🔎 *Da cercare:* `pandas isna sum`

In [ ]:
df.isna().sum()

Domanda: perché **non** è una buona idea riempire i valori mancanti con 0?

*Risposta:* Perché 0 è un valore, non un'assenza: un'impresa con 0 addetti o 0 fatturato abbasserebbe le medie e falserebbe ogni analisi. Un valore mancante invece viene escluso dai calcoli.

## Domanda finale

Pulire i dati vuol dire **prendere decisioni**. Elenca le decisioni che hai preso oggi (cosa hai cambiato e perché). In un lavoro vero, vanno scritte da qualche parte.

*Risposta:* 1) eliminate 15 righe duplicate; 2) fatturato convertito in numero, "n.d." come mancanti; 3) 5 fatturati oltre 200.000 divisi per 1.000 (errore di unità); 4) regioni uniformate (20); 5) addetti negativi di IMP0253 (2019) corretti a 12 su indicazione del collega; 6) valori mancanti lasciati mancanti.

---
## ⭐ Bonus (per chi ha finito)

### B1. Le date

La colonna `data_costituzione` è testo, e le date sono scritte in **due formati diversi**. Convertila in vere date. Verifica che nessuna data sia andata persa.

🔎 *Da cercare:* `pandas to_datetime format mixed dayfirst`

In [ ]:
df["data_costituzione"] = pd.to_datetime(df["data_costituzione"], format="mixed", dayfirst=True)
df["data_costituzione"].isna().sum()

### B2. Salva il lavoro

Salva il dataset pulito in un nuovo file Excel, `imprese_pulito.xlsx`, senza la colonna dell'indice. Poi scaricalo dal pannello dei file di Colab (icona della cartella a sinistra).

🔎 *Da cercare:* `pandas to_excel index`

In [ ]:
df.to_excel("imprese_pulito.xlsx", index=False)

---
## Compito a casa

- [Kaggle Learn – Pandas](https://www.kaggle.com/learn/pandas), lezione **Data Types and Missing Values** (~20 min)
- Facoltativo: [QuantEcon – Pandas](https://python-programming.quantecon.org/pandas.html), la parte sulla pulizia dei dati